# Conjunto de datos completo sin clusterización

In [1]:
#Importaciones
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler

#Lectura de datos
datos = pd.read_excel('03_Clusterizacion.xlsx')
datos.head(24)

,Fecha,Generación,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Cluster KMeans,Cluster GMM
0,2022-09-01 00:00:00,0.000000,19,7,77,0,4,15,0,Noche,Noche
1,2022-09-01 01:00:00,0.000000,19,7,82,0,4,16,1,Noche,Noche
2,2022-09-01 02:00:00,0.000000,18,9,85,0,3,16,2,Noche,Noche
3,2022-09-01 03:00:00,0.000000,18,11,87,0,3,16,3,Noche,Noche
4,2022-09-01 04:00:00,0.000000,18,11,88,0,3,16,4,Noche,Noche
5,2022-09-01 05:00:00,0.000000,17,15,86,0,3,14,5,Noche,Noche
6,2022-09-01 06:00:00,0.000000,18,47,89,0,3,16,6,Nublado,Lluvioso
7,2022-09-01 07:00:00,6.584959,18,51,95,0,4,17,7,Nublado,Lluvioso
8,2022-09-01 08:00:00,560.422022,18,47,100,0,3,18,8,Nublado,Lluvioso
9,2022-09-01 09:00:00,7720.582326,18,5,100,1,4,18,9,Nublado,Lluvioso


In [2]:
datos["Generacion_prev_hour"] = datos["Generación"].shift(1)
datos["Generacion_prev_day"] = datos["Generación"].shift(24)
datos = datos.dropna(how="any", axis= 0)

Definimos X y y

In [3]:
datos_dia = datos[datos["Cluster GMM"] == "Nublado"].copy()
datos_dia.head(10)

,Fecha,Generación,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Cluster KMeans,Cluster GMM,Generacion_prev_hour,Generacion_prev_day
35,2022-09-02 11:00:00,17036.043251,20,0,71,2,2,14,11,Soleado,Nublado,5030.740421,22189.147406
36,2022-09-02 12:00:00,27523.885172,21,0,57,4,2,13,12,Soleado,Nublado,17036.043251,29196.986647
227,2022-09-10 11:00:00,26530.520060,19,0,74,4,1,14,11,Soleado,Nublado,16754.858938,23013.622622
228,2022-09-10 12:00:00,28976.805233,20,0,62,7,1,13,12,Soleado,Nublado,26530.520060,16313.782163
275,2022-09-12 11:00:00,27101.921674,18,0,82,4,2,15,11,Soleado,Nublado,21597.209964,23484.596732
276,2022-09-12 12:00:00,30000.000000,19,0,69,7,2,13,12,Soleado,Nublado,27101.921674,22800.000000
277,2022-09-12 13:00:00,27281.964164,21,0,58,10,2,12,13,Soleado,Nublado,30000.000000,21600.000000
299,2022-09-13 11:00:00,17821.243227,17,0,72,4,2,12,11,Soleado,Nublado,21580.173881,27101.921674
300,2022-09-13 12:00:00,19421.129165,19,0,61,7,2,12,12,Soleado,Nublado,17821.243227,30000.000000
491,2022-09-21 11:00:00,17160.408876,18,0,76,4,2,13,11,Soleado,Nublado,23029.146524,12327.870821


In [4]:
columns = datos_dia.drop(columns=["Fecha", "Generación", "Cluster KMeans", "Cluster GMM"]).columns

In [5]:
X = datos_dia[columns]
X

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day
35,20,0,71,2,2,14,11,5030.740421,22189.147406
36,21,0,57,4,2,13,12,17036.043251,29196.986647
227,19,0,74,4,1,14,11,16754.858938,23013.622622
228,20,0,62,7,1,13,12,26530.520060,16313.782163
275,18,0,82,4,2,15,11,21597.209964,23484.596732
...,...,...,...,...,...,...,...,...,...
18250,14,0,93,2,1,13,9,7302.000000,17824.000000
18251,17,0,78,4,1,13,10,18014.000000,23097.000000
18252,20,0,64,6,1,13,11,23010.000000,26140.000000
18285,22,0,45,0,1,9,20,1450.000000,0.000000


In [6]:
y = datos_dia[['Generación']]
y

,Generación
35,17036.043251
36,27523.885172
227,26530.520060
228,28976.805233
275,27101.921674
...,...
18250,18014.000000
18251,23010.000000
18252,26156.000000
18285,0.000000


Dividimos entrenamiento, validación y prueba

In [7]:
train_size = int(0.7 * len(X))
val_size = int(0.85 * len(X))

In [8]:
# Entrenamiento, validación y prueba, 75, 15 y 15
X_train, y_train =  X.iloc[:train_size, :], y.iloc[:train_size, :]
X_val, y_val = X.iloc[train_size:val_size, :], y.iloc[train_size:val_size, :]
X_test, y_test = X.iloc[val_size:, :],  y.iloc[val_size:,:]

print(f'X_train: {len(X_train)}, y_train: {len(y_train)}')
print(f'X_val: {len(X_val)}, y_val: {len(y_val)}')
print(f'X_test: {len(X_test)}, y_test: {len(y_test)}')

X_train: 1847, y_train: 1847
X_val: 396, y_val: 396
X_test: 396, y_test: 396


## Escalar con MinMaxScaler

In [9]:
from sklearn.preprocessing import MinMaxScaler

In [10]:
x_scaler = MinMaxScaler().fit(X_train)
x_scaler

MinMaxScaler()

In [11]:
X_train_scaled = x_scaler.transform(X_train)
print(X_train_scaled)
print(X_train_scaled.shape)

[[0.52631579 0.         0.69473684 ... 0.33333333 0.16769135 0.73963825]
 [0.55263158 0.         0.54736842 ... 0.4        0.56786811 0.97323289]
 [0.5        0.         0.72631579 ... 0.33333333 0.5584953  0.76712075]
 ...
 [0.44736842 0.         0.25263158 ... 1.         0.         0.        ]
 [0.18421053 0.         0.72631579 ... 0.         0.         0.        ]
 [0.18421053 0.         0.75789474 ... 0.06666667 0.         0.        ]]
(1847, 9)


In [12]:
X_train_scaled_df = pd.DataFrame(X_train_scaled, index=X_train.index, columns=X_train.columns)
X_train_scaled_df

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day
35,0.526316,0.0,0.694737,0.153846,0.333333,0.777778,0.333333,0.167691,0.739638
36,0.552632,0.0,0.547368,0.307692,0.333333,0.722222,0.400000,0.567868,0.973233
227,0.500000,0.0,0.726316,0.307692,0.000000,0.777778,0.333333,0.558495,0.767121
228,0.526316,0.0,0.600000,0.538462,0.000000,0.722222,0.400000,0.884351,0.543793
275,0.473684,0.0,0.810526,0.307692,0.333333,0.833333,0.333333,0.719907,0.782820
...,...,...,...,...,...,...,...,...,...
11733,0.421053,0.0,0.473684,0.000000,0.333333,0.277778,0.933333,0.036533,0.000000
11757,0.500000,0.0,0.210526,0.000000,0.666667,0.055556,0.933333,0.030500,0.000000
11758,0.447368,0.0,0.252632,0.000000,0.333333,0.055556,1.000000,0.000000,0.000000
11767,0.184211,0.0,0.726316,0.000000,0.000000,0.166667,0.000000,0.000000,0.000000


In [13]:
X_val_scaled = x_scaler.transform(X_val)
print(X_val_scaled)
print(X_val_scaled.shape)

[[0.13157895 0.         0.84210526 ... 0.13333333 0.         0.00983333]
 [0.18421053 0.         0.78947368 ... 0.2        0.02443333 0.18196667]
 [0.26315789 0.         0.64210526 ... 0.26666667 0.53976667 0.31103333]
 ...
 [0.18421053 0.         0.90526316 ... 0.13333333 0.043      0.34236667]
 [0.23684211 0.         0.85263158 ... 0.2        0.39413333 0.81946667]
 [0.34210526 0.         0.65263158 ... 0.26666667 0.84823333 0.9256    ]]
(396, 9)


In [14]:
X_val_scaled_df = pd.DataFrame(X_val_scaled, index=X_val.index, columns=X_val.columns)
X_val_scaled_df

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day
11769,0.131579,0.0,0.842105,0.000000,0.0,0.166667,0.133333,0.000000,0.009833
11770,0.184211,0.0,0.789474,0.076923,0.0,0.222222,0.200000,0.024433,0.181967
11771,0.263158,0.0,0.642105,0.153846,0.0,0.222222,0.266667,0.539767,0.311033
11772,0.368421,0.0,0.463158,0.307692,0.0,0.222222,0.333333,0.914833,0.793067
11773,0.473684,0.0,0.326316,0.307692,0.0,0.166667,0.400000,0.929333,0.853733
...,...,...,...,...,...,...,...,...,...
13951,0.210526,0.0,0.852632,0.000000,0.0,0.333333,0.000000,0.000000,0.000000
13952,0.184211,0.0,0.936842,0.000000,0.0,0.333333,0.066667,0.000000,0.013300
13953,0.184211,0.0,0.905263,0.076923,0.0,0.333333,0.133333,0.043000,0.342367
13954,0.236842,0.0,0.852632,0.230769,0.0,0.333333,0.200000,0.394133,0.819467


In [15]:
X_test_scaled = x_scaler.transform(X_test)
print(X_test_scaled)
print(X_test_scaled.shape)

[[0.44736842 0.         0.45263158 ... 0.33333333 0.97296667 0.94383333]
 [0.71052632 0.         0.08421053 ... 0.6        0.98766667 0.93226667]
 [0.76315789 0.         0.05263158 ... 0.66666667 0.96936667 0.91333333]
 ...
 [0.52631579 0.         0.62105263 ... 0.33333333 0.767      0.87133333]
 [0.57894737 0.         0.42105263 ... 0.93333333 0.04833333 0.        ]
 [0.52631579 0.         0.51578947 ... 1.         0.         0.        ]]
(396, 9)


In [16]:
X_test_scaled_df = pd.DataFrame(X_test_scaled, index=X_test.index, columns=X_test.columns)
X_test_scaled_df

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day
13956,0.447368,0.0,0.452632,0.538462,0.0,0.333333,0.333333,0.972967,0.943833
13960,0.710526,0.0,0.084211,0.538462,0.0,0.166667,0.600000,0.987667,0.932267
13961,0.763158,0.0,0.052632,0.384615,0.0,0.277778,0.666667,0.969367,0.913333
13962,0.815789,0.0,0.042105,0.230769,0.0,0.277778,0.733333,0.955300,0.877900
13963,0.763158,0.0,0.073684,0.076923,0.0,0.166667,0.800000,0.880333,0.708767
...,...,...,...,...,...,...,...,...,...
18250,0.368421,0.0,0.926316,0.153846,0.0,0.722222,0.200000,0.243400,0.594133
18251,0.447368,0.0,0.768421,0.307692,0.0,0.722222,0.266667,0.600467,0.769900
18252,0.526316,0.0,0.621053,0.461538,0.0,0.722222,0.333333,0.767000,0.871333
18285,0.578947,0.0,0.421053,0.000000,0.0,0.500000,0.933333,0.048333,0.000000


In [17]:
x_scaller_all = MinMaxScaler().fit(X)
print(x_scaller_all)

MinMaxScaler()


In [18]:
X_scaled = x_scaller_all.transform(X)
print(X_scaled)
print(X_scaled.shape)

[[0.51282051 0.         0.70103093 ... 0.33333333 0.16769135 0.73963825]
 [0.53846154 0.         0.55670103 ... 0.4        0.56786811 0.97323289]
 [0.48717949 0.         0.73195876 ... 0.33333333 0.5584953  0.76712075]
 ...
 [0.51282051 0.         0.62886598 ... 0.33333333 0.767      0.87133333]
 [0.56410256 0.         0.43298969 ... 0.93333333 0.04833333 0.        ]
 [0.51282051 0.         0.5257732  ... 1.         0.         0.        ]]
(2639, 9)


In [19]:
X_scaled_df = pd.DataFrame(X_scaled, index=X.index, columns=X.columns)
X_scaled_df

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day
35,0.512821,0.0,0.701031,0.153846,0.333333,0.777778,0.333333,0.167691,0.739638
36,0.538462,0.0,0.556701,0.307692,0.333333,0.722222,0.400000,0.567868,0.973233
227,0.487179,0.0,0.731959,0.307692,0.000000,0.777778,0.333333,0.558495,0.767121
228,0.512821,0.0,0.608247,0.538462,0.000000,0.722222,0.400000,0.884351,0.543793
275,0.461538,0.0,0.814433,0.307692,0.333333,0.833333,0.333333,0.719907,0.782820
...,...,...,...,...,...,...,...,...,...
18250,0.358974,0.0,0.927835,0.153846,0.000000,0.722222,0.200000,0.243400,0.594133
18251,0.435897,0.0,0.773196,0.307692,0.000000,0.722222,0.266667,0.600467,0.769900
18252,0.512821,0.0,0.628866,0.461538,0.000000,0.722222,0.333333,0.767000,0.871333
18285,0.564103,0.0,0.432990,0.000000,0.000000,0.500000,0.933333,0.048333,0.000000


In [20]:
y_scaler = MinMaxScaler().fit(y_train)
print(y_scaler)

MinMaxScaler()


In [21]:
y_train_scaled = y_scaler.transform(y_train)
print(y_train_scaled)
print(y_train_scaled.shape)

[[0.56786811]
 [0.91746284]
 [0.88435067]
 ...
 [0.        ]
 [0.        ]
 [0.        ]]
(1847, 1)


In [22]:
y_train_scaled_df = pd.DataFrame(y_train_scaled, index=y_train.index, columns=y_train.columns)
y_train_scaled_df

,Generación
35,0.567868
36,0.917463
227,0.884351
228,0.965894
275,0.903397
...,...
11733,0.000000
11757,0.000000
11758,0.000000
11767,0.000000


In [23]:
y_val_scaled = y_scaler.transform(y_val)
print(y_val_scaled)
print(y_val_scaled.shape)

[[2.44333333e-02]
 [5.39766667e-01]
 [9.14833333e-01]
 [9.29333333e-01]
 [9.46266667e-01]
 [8.36400000e-01]
 [8.03333333e-01]
 [8.37600000e-01]
 [8.36633333e-01]
 [8.09166667e-01]
 [8.36766667e-01]
 [8.12266667e-01]
 [4.06900000e-01]
 [3.18666667e-02]
 [8.06933333e-01]
 [8.85000000e-01]
 [8.08366667e-01]
 [3.55366667e-01]
 [8.36400000e-01]
 [8.50366667e-01]
 [8.51533333e-01]
 [8.37533333e-01]
 [8.05866667e-01]
 [8.36866667e-01]
 [8.09666667e-01]
 [4.06400000e-01]
 [3.15333333e-02]
 [3.15333333e-02]
 [0.00000000e+00]
 [0.00000000e+00]
 [9.34133333e-01]
 [9.49200000e-01]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [2.45333333e-02]
 [5.35233333e-01]
 [9.49400000e-01]
 [9.30866667e-01]
 [9.48733333e-01]
 [9.43500000e-01]
 [9.33433333e-01]
 [8.93566667e-01]
 [9.29733333e-01]
 [8.98200000e-01]
 [4.51566667e-01]
 [3.57666667e-02]
 [0.00000000e+00]
 [8.30000000e-03]
 [0.00000000e+00]
 [0.00000000e+00]
 [2.54233333e-01]
 [7.31700000e-01]
 [7.93966667e-01]
 [0.000000

In [24]:
y_val_scaled_df = pd.DataFrame(y_val_scaled, index=y_val.index, columns=y_val.columns)
y_val_scaled_df

,Generación
11769,0.024433
11770,0.539767
11771,0.914833
11772,0.929333
11773,0.946267
...,...
13951,0.000000
13952,0.043000
13953,0.394133
13954,0.848233


In [25]:
y_test_scaled = y_scaler.transform(y_test)
print(y_test_scaled)
print(y_test_scaled.shape)

[[9.95466667e-01]
 [9.69366667e-01]
 [9.55300000e-01]
 [8.80333333e-01]
 [5.72166667e-01]
 [7.90133333e-01]
 [0.00000000e+00]
 [1.43000000e-02]
 [2.74900000e-01]
 [6.55566667e-01]
 [1.53333333e-03]
 [2.79866667e-01]
 [6.67100000e-01]
 [7.40466667e-01]
 [6.19600000e-01]
 [5.03100000e-01]
 [1.34666667e-02]
 [3.42366667e-01]
 [8.34100000e-01]
 [9.25600000e-01]
 [9.35966667e-01]
 [9.50600000e-01]
 [9.28233333e-01]
 [9.34933333e-01]
 [9.27933333e-01]
 [0.00000000e+00]
 [1.81666667e-02]
 [3.80700000e-01]
 [8.66833333e-01]
 [9.74500000e-01]
 [9.95466667e-01]
 [9.95433333e-01]
 [9.94366667e-01]
 [9.85700000e-01]
 [9.95466667e-01]
 [9.61066667e-01]
 [8.29166667e-01]
 [5.44333333e-01]
 [1.08133333e-01]
 [3.33333333e-05]
 [0.00000000e+00]
 [0.00000000e+00]
 [1.35666667e-02]
 [3.43266667e-01]
 [8.19466667e-01]
 [9.25600000e-01]
 [0.00000000e+00]
 [6.71400000e-01]
 [7.48566667e-01]
 [8.18700000e-01]
 [0.00000000e+00]
 [1.19666667e-02]
 [3.10700000e-01]
 [7.17966667e-01]
 [7.62200000e-01]
 [7.969000

In [26]:
y_test_scaled_df = pd.DataFrame(y_test_scaled, index=y_test.index, columns=y_test.columns)
y_test_scaled_df

,Generación
13956,0.995467
13960,0.969367
13961,0.955300
13962,0.880333
13963,0.572167
...,...
18250,0.600467
18251,0.767000
18252,0.871867
18285,0.000000


In [27]:
y_scaller_all = MinMaxScaler().fit(y)
print(y_scaller_all)

MinMaxScaler()


In [28]:
y_scaled = y_scaller_all.transform(y)
print(y_scaled)
print(y_scaled.shape)

[[0.56786811]
 [0.91746284]
 [0.88435067]
 ...
 [0.87186667]
 [0.        ]
 [0.        ]]
(2639, 1)


In [29]:
y_scaled_df = pd.DataFrame(y_scaled, index=y.index, columns=y.columns)
y_scaled_df

,Generación
35,0.567868
36,0.917463
227,0.884351
228,0.965894
275,0.903397
...,...
18250,0.600467
18251,0.767000
18252,0.871867
18285,0.000000


## Definición de modelos

### RandomForest

In [30]:
from lightgbm import LGBMRegressor
import optuna
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
import seaborn as sns
from sklearn.metrics import mean_absolute_percentage_error as mean_absolute_percentage_error
from sklearn.metrics import mean_absolute_error as mean_absolute_error
from sklearn.metrics import mean_squared_error as mean_squared_error
from sklearn.metrics import r2_score as r2_score

In [31]:
# Inicializar listas para métricas
LightGBM_model = LGBMRegressor(num_leaves=500, subsample= 0.10698460631792395, colsample_bytree= 0.7272836809565294, min_data_in_leaf= 85)
LightGBM_model.fit(X_train_scaled_df, y_train_scaled_df)
resultados = pd.DataFrame(index = y_test_scaled_df.index, columns=["LightGBM"])
#Ciclo diario de predicción
for i in range(len(X_test)):
    inicio = i * 1
    fin = inicio + 1

    X_test_seg = X_test_scaled_df.iloc[inicio:fin, :]
    y_test_seg = y_test_scaled_df.iloc[inicio:fin]

    if len(X_test_seg) < 1:
        break

    y_pred = LightGBM_model.predict(X_test_seg)
    y_pred = y_scaler.inverse_transform(y_pred.reshape(-1, 1))
    y_pred = np.clip(y_pred, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

    resultados.iloc[i, 0] = y_pred[0, 0]

[LightGBM] [Warning] min_data_in_leaf is set=85, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=85
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] min_data_in_leaf is set=85, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=85
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000397 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 696
[LightGBM] [Info] Number of data points in the train set: 1847, number of used features: 8
[LightGBM] [Info] Start training from score 0.454230
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further split

In [32]:
resultados

,LightGBM
13956,29802.736495
13960,29095.19111
13961,29136.770699
13962,28333.916793
13963,18488.082907
...,...
18250,21351.153925
18251,22831.557606
18252,24912.539517
18285,0.0


In [33]:
predicciones = y_test.copy()
predicciones

,Generación
13956,29864.0
13960,29081.0
13961,28659.0
13962,26410.0
13963,17165.0
...,...
18250,18014.0
18251,23010.0
18252,26156.0
18285,0.0


In [34]:
predicciones["LightGBM"] = resultados["LightGBM"]
predicciones

,Generación,LightGBM
13956,29864.0,29802.736495
13960,29081.0,29095.19111
13961,28659.0,29136.770699
13962,26410.0,28333.916793
13963,17165.0,18488.082907
...,...,...
18250,18014.0,21351.153925
18251,23010.0,22831.557606
18252,26156.0,24912.539517
18285,0.0,0.0


In [35]:
print(f"MAE: {mean_absolute_error(predicciones['Generación'], predicciones['LightGBM']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones['Generación'], predicciones['LightGBM'])):.4f}")
print(f"R²: {r2_score(predicciones['Generación'], predicciones['LightGBM']):.4f}")

MAE: 1180.6107
RMSE: 1838.9254
R²: 0.9682


## Random Forest

In [36]:
from sklearn.ensemble import RandomForestRegressor

In [37]:
#Modelo LightGBM
RF_model = RandomForestRegressor(
    criterion="squared_error",
    random_state=0,
    n_estimators=400,
    min_impurity_decrease=0,
    max_depth=None,
    bootstrap=True
)
RF_model.fit(X_train_scaled_df, y_train_scaled_df)
# Inicializar listas para métricas
resultados = pd.DataFrame(index = y_test_scaled_df.index, columns=["Random Forest"])
#Ciclo diario de predicción
for i in range(len(X_test)):
    inicio = i * 1
    fin = inicio + 1

    X_test_seg = X_test_scaled_df.iloc[inicio:fin, :]
    y_test_seg = y_test_scaled_df.iloc[inicio:fin]

    if len(X_test_seg) < 1:
        break

    y_pred = RF_model.predict(X_test_seg)
    y_pred = y_scaler.inverse_transform(y_pred.reshape(-1, 1))
    y_pred = np.clip(y_pred, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

    resultados.iloc[i, 0] = y_pred[0, 0]

In [38]:
predicciones["Random Forest"] = resultados["Random Forest"]
predicciones

,Generación,LightGBM,Random Forest
13956,29864.0,29802.736495,29306.46
13960,29081.0,29095.19111,29272.7725
13961,28659.0,29136.770699,29224.675
13962,26410.0,28333.916793,28923.955
13963,17165.0,18488.082907,20188.37
...,...,...,...
18250,18014.0,21351.153925,20554.05
18251,23010.0,22831.557606,24302.920607
18252,26156.0,24912.539517,25643.327875
18285,0.0,0.0,0.0


## Preparación redes neuronales

In [39]:
import numpy as np
import pandas as pd

def create_sliding_window_with_index(data_X, data_y, lookback):
    X, y, indices = [], [], []
    
    # Asegurar que `data_y` tiene los mismos índices que `data_X`
    data_y = data_y.reindex(data_X.index)

    max_index = len(data_X) - lookback

    for i in range(max_index):
        X.append(data_X.iloc[i:i + lookback].values)  # Ventana de entrada
        
        # Obtener el índice correcto en `data_y`
        y_index = data_X.index[i + lookback]

        # Extraer el valor correspondiente de `data_y`
        if y_index in data_y.index:
            y_value = data_y.loc[y_index]
            if isinstance(y_value, pd.Series):  # Si devuelve una serie, extraer el valor
                y_value = y_value.iloc[0]
        else:
            y_value = np.nan  # Si no está, asignamos NaN

        y.append(y_value)
        indices.append(y_index)  # 🔹 Guardamos el índice original de `data_y`

    # Convertimos `X` en un array y `y` en DataFrame conservando sus índices originales
    X_array = np.array(X)
    y_df = pd.DataFrame(y, index=indices, columns=['y'])  # 🔹 Conservamos los índices originales

    return X_array, y_df


In [40]:
lookback = 48  # Puedes ajustar a 24, 72, etc.

# Aplicar la ventana deslizante a cada conjunto
X_train_windowed, y_train_windowed = create_sliding_window_with_index(X_train_scaled_df, y_train_scaled_df, lookback)
X_val_windowed, y_val_windowed = create_sliding_window_with_index(X_val_scaled_df, y_val_scaled_df, lookback)
X_test_windowed, y_test_windowed = create_sliding_window_with_index(X_test_scaled_df, y_test_scaled_df, lookback)


In [41]:
print(f'X_train: {X_train_windowed.shape}, y_train: {y_train_windowed.shape}')
print(f'X_val: {X_val_windowed.shape}, y_val: {y_val_windowed.shape}')
print(f'X_test: {X_test_windowed.shape}, y_test: {y_test_windowed.shape}')

X_train: (1799, 48, 9), y_train: (1799, 1)
X_val: (348, 48, 9), y_val: (348, 1)
X_test: (348, 48, 9), y_test: (348, 1)


## CTNET

In [42]:
import tensorflow as tf
from tensorflow.keras import layers

In [43]:
def compile_and_fit(model, xtrain=X_train_windowed, ytrain=y_train_windowed, learning_rate=0.0001):
    model.compile(loss=[tf.keras.losses.MeanSquaredError()],
                  optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                  metrics=[tf.keras.metrics.RootMeanSquaredError(), tf.keras.metrics.MeanAbsolutePercentageError(), tf.keras.metrics.MeanAbsoluteError()])
    
    history = model.fit(xtrain, ytrain, epochs=50,
                        batch_size=512, validation_split=0.2, verbose=1)
    return history

def Loss(train_loss, valid_loss):
    plt.plot(train_loss)
    plt.plot(valid_loss)
    plt.rcParams["figure.figsize"] = (15, 3)
    plt.title('Model Losses')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend(['Train Loss', 'Validation Loss'], loc='upper left')
    plt.savefig('out/loss_plot.png')
    plt.show()

def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0):
    x = layers.LayerNormalization()(inputs)
    x = layers.Conv1D(filters=ff_dim, kernel_size=1, activation="relu", padding="same")(x)
    x = layers.Conv1D(filters=128, kernel_size=2, activation="relu", padding="same")(x)
    x = layers.Conv1D(filters=inputs.shape[-1], kernel_size=1)(x)
    res = x + inputs
    norm_x = layers.LayerNormalization()(res)
    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(norm_x, norm_x)
    res = x + inputs
    norm_x = layers.LayerNormalization()(res)
    return norm_x

def build_model(input_shape, head_size, num_heads, ff_dim, num_transformer_blocks, mlp_units, dropout=0, mlp_dropout=0):
    inputs = tf.keras.Input(shape=input_shape)
    x = inputs
    
    for _ in range(num_transformer_blocks):
        enc_out = transformer_encoder(x, head_size, num_heads, ff_dim, dropout)
    
    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(enc_out, enc_out)
    res = x + enc_out
    x = layers.LayerNormalization(epsilon=1e-6)(res)
    x = layers.GlobalAveragePooling1D(data_format="channels_first")(x)
    x = layers.Dense(832, activation="relu")(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(mlp_dropout)(x)

    outputs = layers.Dense(1)(x)
    
    return tf.keras.Model(inputs, outputs)

In [44]:
CTNET = build_model((X_train_windowed.shape[1], X_train_windowed.shape[2]), head_size=4, num_heads=3, ff_dim=32, num_transformer_blocks=3, mlp_units=[256], mlp_dropout=0.3, dropout=0.2)

In [45]:
history = compile_and_fit(CTNET)

Epoch 1/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 56s 2s/step - loss: 0.3589 - mean_absolute_error: 0.4593 - mean_absolute_percentage_error: 164553.2500 - root_mean_squared_error: 0.5991 - val_loss: 0.3000 - val_mean_absolute_error: 0.4022 - val_mean_absolute_percentage_error: 1593694.6250 - val_root_mean_squared_error: 0.5477
Epoch 2/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 804ms/step - loss: 0.3572 - mean_absolute_error: 0.4576 - mean_absolute_percentage_error: 1157786.2500 - root_mean_squared_error: 0.5977 - val_loss: 0.2955 - val_mean_absolute_error: 0.4007 - val_mean_absolute_percentage_error: 3351443.2500 - val_root_mean_squared_error: 0.5436
Epoch 3/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 427ms/step - loss: 0.3476 - mean_absolute_error: 0.4508 - mean_absolute_percentage_error: 2253094.5000 - root_mean_squared_error: 0.5895 - val_loss: 0.2906 - val_mean_absolute_error: 0.3991 - val_mean_absolute_percentage_error: 5255581.0000 - val_root_mean_squared_error: 0.5391
Epoch 4/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 479ms/step

In [46]:
CTNET_predictions = CTNET.predict(X_test_windowed)
CTNET_predictions

11/11 ━━━━━━━━━━━━━━━━━━━━ 6s 313ms/step


array([[0.6225866 ],
       [0.6233591 ],
       [0.62186766],
       [0.6111368 ],
       [0.6027219 ],
       [0.5947629 ],
       [0.5947823 ],
       [0.60276794],
       [0.6079874 ],
       [0.6095917 ],
       [0.6050049 ],
       [0.6135638 ],
       [0.619771  ],
       [0.62020206],
       [0.6221294 ],
       [0.6245698 ],
       [0.6250838 ],
       [0.6237534 ],
       [0.6192645 ],
       [0.6125192 ],
       [0.6021797 ],
       [0.593387  ],
       [0.59552056],
       [0.5996876 ],
       [0.6002264 ],
       [0.60282457],
       [0.61459076],
       [0.60892594],
       [0.60789967],
       [0.6073985 ],
       [0.6086134 ],
       [0.60723835],
       [0.60607016],
       [0.6013486 ],
       [0.59059244],
       [0.5723063 ],
       [0.5556365 ],
       [0.5594185 ],
       [0.56825876],
       [0.5832755 ],
       [0.60320747],
       [0.61356807],
       [0.6208199 ],
       [0.6212069 ],
       [0.61301637],
       [0.6066049 ],
       [0.603166  ],
       [0.610

In [47]:
CTNET_predictions = y_scaler.inverse_transform(CTNET_predictions.reshape(-1, 1))
CTNET_predictions = np.clip(CTNET_predictions, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

In [48]:
resultados = pd.DataFrame(CTNET_predictions, index = y_test_windowed.index, columns=["CTNET"])

In [49]:
predicciones["CTNET"] = resultados["CTNET"]
predicciones

,Generación,LightGBM,Random Forest,CTNET
13956,29864.0,29802.736495,29306.46,NaN
13960,29081.0,29095.19111,29272.7725,NaN
13961,28659.0,29136.770699,29224.675,NaN
13962,26410.0,28333.916793,28923.955,NaN
13963,17165.0,18488.082907,20188.37,NaN
...,...,...,...,...
18250,18014.0,21351.153925,20554.05,19111.265625
18251,23010.0,22831.557606,24302.920607,19171.068359
18252,26156.0,24912.539517,25643.327875,19269.224609
18285,0.0,0.0,0.0,19271.185547


In [50]:
predicciones["CTNET"] = predicciones["CTNET"].fillna(0)

In [51]:
# import optuna
# import tensorflow as tf
# from tensorflow.keras import layers
# from sklearn.model_selection import train_test_split

# # Definir la función objetivo para Optuna
# def objective(trial):
#     # Sugerir valores para los hiperparámetros
#     head_size = trial.suggest_int("head_size", 8, 64, step=8)
#     num_heads = trial.suggest_int("num_heads", 2, 8, step=2)
#     ff_dim = trial.suggest_int("ff_dim", 32, 256, step=32)
#     num_transformer_blocks = trial.suggest_int("num_transformer_blocks", 1, 4)
#     mlp_units = trial.suggest_categorical("mlp_units", [[128, 64], [256, 128, 64], [512, 256, 128]])
#     dropout = trial.suggest_float("dropout", 0.1, 0.5, step=0.1)
#     mlp_dropout = trial.suggest_float("mlp_dropout", 0.1, 0.5, step=0.1)
#     learning_rate = trial.suggest_loguniform("learning_rate", 1e-5, 1e-2)

#     # Construcción del modelo con los hiperparámetros sugeridos
#     model = build_model(
#         input_shape=X_train_windowed.shape[1:],
#         head_size=head_size,
#         num_heads=num_heads,
#         ff_dim=ff_dim,
#         num_transformer_blocks=num_transformer_blocks,
#         mlp_units=mlp_units,
#         dropout=dropout,
#         mlp_dropout=mlp_dropout
#     )

#     # Compilar el modelo con los hiperparámetros sugeridos
#     model.compile(
#         loss=tf.keras.losses.MeanSquaredError(),
#         optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
#         metrics=[tf.keras.metrics.RootMeanSquaredError()]
#     )

#     # Entrenamiento con un número reducido de épocas para acelerar la búsqueda
#     history = model.fit(
#         X_train_windowed, y_train_windowed,
#         validation_split=0.2,
#         epochs=50,  # Reducimos las épocas para acelerar la búsqueda
#         batch_size=512,
#         verbose=0
#     )

#     # Obtener la métrica de validación (RMSE) y minimizarla
#     val_rmse = min(history.history["val_root_mean_squared_error"])
    
#     return val_rmse  # Queremos minimizar el RMSE

# # Ejecutar la optimización de hiperparámetros
# study = optuna.create_study(direction="minimize")
# study.optimize(objective, n_trials=20, timeout=3600)  # 20 iteraciones, máximo 1 hora

# # Mostrar los mejores hiperparámetros encontrados
# best_params = study.best_params
# print(f"Mejores hiperparámetros: {best_params}")


In [52]:
CTNET = build_model((X_train_windowed.shape[1], X_train_windowed.shape[2]), head_size=16, num_heads=8, ff_dim=256, num_transformer_blocks=1, mlp_units=[128,64], mlp_dropout=0.2, dropout=0.2)

In [53]:
history = compile_and_fit(CTNET, learning_rate = 0.0010762230908145116)

Epoch 1/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 58s 6s/step - loss: 0.3495 - mean_absolute_error: 0.4538 - mean_absolute_percentage_error: 2203426.2500 - root_mean_squared_error: 0.5912 - val_loss: 0.2467 - val_mean_absolute_error: 0.3849 - val_mean_absolute_percentage_error: 24101522.0000 - val_root_mean_squared_error: 0.4967
Epoch 2/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 11s 4s/step - loss: 0.2783 - mean_absolute_error: 0.4162 - mean_absolute_percentage_error: 19725554.0000 - root_mean_squared_error: 0.5275 - val_loss: 0.1655 - val_mean_absolute_error: 0.3560 - val_mean_absolute_percentage_error: 75683224.0000 - val_root_mean_squared_error: 0.4068
Epoch 3/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 17s 2s/step - loss: 0.1848 - mean_absolute_error: 0.3722 - mean_absolute_percentage_error: 55936064.0000 - root_mean_squared_error: 0.4297 - val_loss: 0.1626 - val_mean_absolute_error: 0.3627 - val_mean_absolute_percentage_error: 168309504.0000 - val_root_mean_squared_error: 0.4032
Epoch 4/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 13s 3s/ste

In [54]:
CTNET_predictions = CTNET.predict(X_test_windowed)
CTNET_predictions

11/11 ━━━━━━━━━━━━━━━━━━━━ 13s 747ms/step


array([[ 1.1772695 ],
       [ 1.1132349 ],
       [ 0.8773222 ],
       [ 0.19340742],
       [ 0.49704927],
       [ 1.0572242 ],
       [ 0.87262917],
       [ 0.88660127],
       [ 0.8634442 ],
       [ 0.7228949 ],
       [ 0.90323937],
       [ 0.94993466],
       [ 0.7129201 ],
       [ 0.79267323],
       [ 0.80215794],
       [ 0.6029861 ],
       [ 0.5697391 ],
       [ 0.2584605 ],
       [ 0.11226144],
       [ 0.12563016],
       [ 0.19609728],
       [ 0.8391011 ],
       [ 0.8746485 ],
       [ 0.5629952 ],
       [ 0.76774806],
       [ 0.9534504 ],
       [ 0.9786733 ],
       [ 0.73523283],
       [ 0.6431557 ],
       [ 0.69563586],
       [ 0.60527086],
       [ 0.34439832],
       [ 0.63822126],
       [ 0.1499213 ],
       [ 0.00137779],
       [ 0.07005148],
       [ 0.02447502],
       [ 0.5958508 ],
       [ 0.76419413],
       [ 0.6745958 ],
       [ 0.79329455],
       [ 0.8788632 ],
       [ 0.83524084],
       [ 0.38943487],
       [ 0.24909346],
       [ 0

In [55]:
CTNET_predictions = y_scaler.inverse_transform(CTNET_predictions.reshape(-1, 1))
CTNET_predictions = np.clip(CTNET_predictions, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

In [56]:
resultados = pd.DataFrame(CTNET_predictions, index = y_test_windowed.index, columns=["CTNET"])

In [57]:
predicciones["CTNET"] = resultados["CTNET"]
predicciones

,Generación,LightGBM,Random Forest,CTNET
13956,29864.0,29802.736495,29306.46,NaN
13960,29081.0,29095.19111,29272.7725,NaN
13961,28659.0,29136.770699,29224.675,NaN
13962,26410.0,28333.916793,28923.955,NaN
13963,17165.0,18488.082907,20188.37,NaN
...,...,...,...,...
18250,18014.0,21351.153925,20554.05,18707.601562
18251,23010.0,22831.557606,24302.920607,23466.720703
18252,26156.0,24912.539517,25643.327875,22903.962891
18285,0.0,0.0,0.0,20785.935547


## Forecasting

In [58]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import *
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.losses import MeanSquaredError
from tensorflow.keras.metrics import RootMeanSquaredError
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.losses import Huber
from tensorflow.keras.callbacks import EarlyStopping

In [59]:
Forecast_model = Sequential()
Forecast_model.add(InputLayer((X_train_windowed.shape[1], X_train_windowed.shape[2])))

#CNN
Forecast_model.add(Conv1D(filters=64, kernel_size=2, padding='same', activation='relu'))
Forecast_model.add(BatchNormalization())  # 🔹 Nueva Normalización aquí
Forecast_model.add(MaxPooling1D(pool_size=2))

#model_Soleado.add(Flatten())
#BiLSTM
Forecast_model.add(Bidirectional(LSTM(128, return_sequences=True)))
Forecast_model.add(Bidirectional(LSTM(64, return_sequences=True)))
Forecast_model.add(Dropout(0.2))  # 🔹 Mayor regularización en BiLSTM
Forecast_model.add(Bidirectional(LSTM(32, return_sequences=False)))

#Normalización y Dropout
Forecast_model.add(BatchNormalization())
Forecast_model.add(Dropout(0.3))

# Capas Densas
Forecast_model.add(Dense(16, activation='relu'))
Forecast_model.add(Dense(1, 'relu'))

Forecast_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_12 (Conv1D)              │ (None, 48, 64)         │         1,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 48, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 24, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 24, 256)        │       197,632 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 24, 128)        │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 24, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, 64)             │        41,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 16)             │         1,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 405,985 (1.55 MB)

 Trainable params: 405,729 (1.55 MB)

 Non-trainable params: 256 (1.00 KB)

In [60]:
cp = ModelCheckpoint('Forcasting_model.keras', save_best_only=True)
Forecast_model.compile(optimizer=Adam(learning_rate=0.0001), loss=Huber(delta=1000), metrics=['mae'])
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [61]:
history = Forecast_model.fit(X_train_windowed, y_train_windowed, validation_data=(X_val_windowed, y_val_windowed), epochs=100, batch_size=8, callbacks=[cp, early_stop])

Epoch 1/100
225/225 ━━━━━━━━━━━━━━━━━━━━ 275s 432ms/step - loss: 0.1825 - mae: 0.4561 - val_loss: 0.1867 - val_mae: 0.5005
Epoch 2/100
225/225 ━━━━━━━━━━━━━━━━━━━━ 120s 320ms/step - loss: 0.1358 - mae: 0.3862 - val_loss: 0.1534 - val_mae: 0.4481
Epoch 3/100
225/225 ━━━━━━━━━━━━━━━━━━━━ 91s 353ms/step - loss: 0.1335 - mae: 0.3837 - val_loss: 0.1061 - val_mae: 0.3661
Epoch 4/100
225/225 ━━━━━━━━━━━━━━━━━━━━ 88s 364ms/step - loss: 0.1104 - mae: 0.3601 - val_loss: 0.0972 - val_mae: 0.3575
Epoch 5/100
225/225 ━━━━━━━━━━━━━━━━━━━━ 79s 326ms/step - loss: 0.1012 - mae: 0.3434 - val_loss: 0.0888 - val_mae: 0.3275
Epoch 6/100
225/225 ━━━━━━━━━━━━━━━━━━━━ 85s 326ms/step - loss: 0.0935 - mae: 0.3349 - val_loss: 0.0838 - val_mae: 0.3364
Epoch 7/100
225/225 ━━━━━━━━━━━━━━━━━━━━ 71s 264ms/step - loss: 0.0999 - mae: 0.3409 - val_loss: 0.0939 - val_mae: 0.3517
Epoch 8/100
225/225 ━━━━━━━━━━━━━━━━━━━━ 98s 327ms/step - loss: 0.0946 - mae: 0.3345 - val_loss: 0.0667 - val_mae: 0.2901
Epoch 9/100
225/225 ━━

In [62]:
Forecast_predictions = Forecast_model.predict(X_test_windowed)
Forecast_predictions

11/11 ━━━━━━━━━━━━━━━━━━━━ 65s 3s/step 


array([[0.8515019 ],
       [0.8372051 ],
       [0.67486995],
       [0.01657102],
       [0.48138645],
       [0.7812376 ],
       [0.8378856 ],
       [0.62010723],
       [0.7674374 ],
       [0.42100522],
       [0.66717607],
       [0.71533084],
       [0.31536287],
       [0.75966674],
       [0.32195222],
       [0.33749318],
       [0.28885466],
       [0.03654558],
       [0.        ],
       [0.        ],
       [0.3148396 ],
       [0.7964461 ],
       [0.7926151 ],
       [0.5308808 ],
       [0.5074903 ],
       [0.40932745],
       [0.6625546 ],
       [0.81275207],
       [0.86142665],
       [0.5542529 ],
       [0.33714327],
       [0.32815197],
       [0.38695842],
       [0.24076217],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.69390225],
       [0.6228549 ],
       [0.74144834],
       [0.6559378 ],
       [0.65239197],
       [0.49136537],
       [0.3324894 ],
       [0.        ],
       [0.07898502],
       [0.30697772],
       [0.737

In [63]:
Forecast_predictions = y_scaler.inverse_transform(Forecast_predictions.reshape(-1, 1))
Forecast_predictions = np.clip(Forecast_predictions, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

In [64]:
Forecast_resultados = pd.DataFrame(Forecast_predictions, index = y_test_windowed.index, columns=["Forecast"])

In [65]:
predicciones["Forecast"] = Forecast_resultados["Forecast"]
predicciones

,Generación,LightGBM,Random Forest,CTNET,Forecast
13956,29864.0,29802.736495,29306.46,NaN,NaN
13960,29081.0,29095.19111,29272.7725,NaN,NaN
13961,28659.0,29136.770699,29224.675,NaN,NaN
13962,26410.0,28333.916793,28923.955,NaN,NaN
13963,17165.0,18488.082907,20188.37,NaN,NaN
...,...,...,...,...,...
18250,18014.0,21351.153925,20554.05,18707.601562,5487.743164
18251,23010.0,22831.557606,24302.920607,23466.720703,19692.076172
18252,26156.0,24912.539517,25643.327875,22903.962891,7768.808105
18285,0.0,0.0,0.0,20785.935547,2999.348389


## Métricas

In [66]:
predicciones.loc[~predicciones['CTNET'].isna(),'Generación']

14123    22457.0
14177    24561.0
14191        0.0
14192      359.0
14241     9321.0
          ...   
18250    18014.0
18251    23010.0
18252    26156.0
18285        0.0
18286        0.0
Name: Generación, Length: 348, dtype: float64

## Photovoltaic

In [67]:
from tensorflow.keras.models import Model
inputs = Input(shape=(X_train_windowed.shape[1], X_train_windowed.shape[2]))

# Primera capa CNN
x = Conv1D(filters=64, kernel_size=4, padding='same', activation='relu')(inputs)
x = MaxPooling1D(pool_size=2)(x)

# Segunda capa CNN
x = Conv1D(filters=128, kernel_size=4, padding='same', activation='relu')(x)
x = MaxPooling1D(pool_size=2)(x)

# Capa BiGRU
x = Bidirectional(GRU(64, return_sequences=True))(x)

# Atención: se define de forma explícita
attention = MultiHeadAttention(num_heads=4, key_dim=128)(x, x)

# Aplanar y agregar Dropout
x = Flatten()(attention)
x = Dropout(0.4)(x)
initializer = tf.keras.initializers.HeNormal()
x = Dense(64, activation="relu", kernel_regularizer=l2(0.01))(x)
x = Dense(32, activation="relu")(x)  # Otra capa intermedia

# Capa de salida
outputs = Dense(1, activation="linear")(x)

# Definir el modelo
Photo_model = Model(inputs=inputs, outputs=outputs)

# Resumen del modelo
Photo_model.summary()

Model: "functional_13"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 48, 9)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_13 (Conv1D)  │ (None, 48, 64)    │      2,368 │ input_layer_3[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_1     │ (None, 24, 64)    │          0 │ conv1d_13[0][0]   │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_14 (Conv1D)  │ (None, 24, 128)   │     32,896 │ max_pooling1d_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_2     │ (None, 12, 128)   │          0 │ conv1d_14[0][0]   │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_3     │ (None, 12, 128)   │     74,496 │ max_pooling1d_2[… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 12, 128)   │    263,808 │ bidirectional_3[… │
│ (MultiHeadAttentio… │                   │            │ bidirectional_3[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 1536)      │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_11          │ (None, 1536)      │          0 │ flatten[0][0]     │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 64)        │     98,368 │ dropout_11[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_11 (Dense)    │ (None, 32)        │      2,080 │ dense_10[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_12 (Dense)    │ (None, 1)         │         33 │ dense_11[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 474,049 (1.81 MB)

 Trainable params: 474,049 (1.81 MB)

 Non-trainable params: 0 (0.00 B)

In [68]:
cp2 = ModelCheckpoint('Photovoltaic_model.keras', save_best_only=True)
Photo_model.compile(optimizer=Adam(learning_rate=0.0001), loss="mean_squared_error", metrics=['mae'])
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

In [69]:
history = Photo_model.fit(
    X_train_windowed, y_train_windowed,
    validation_data=(X_val_windowed, y_val_windowed),
    epochs=50,
    batch_size=16,
    callbacks=[cp, early_stop]
)

Epoch 1/50
113/113 ━━━━━━━━━━━━━━━━━━━━ 338s 482ms/step - loss: 1.2916 - mae: 0.3610 - val_loss: 0.9644 - val_mae: 0.3835
Epoch 2/50
113/113 ━━━━━━━━━━━━━━━━━━━━ 46s 323ms/step - loss: 0.8491 - mae: 0.3246 - val_loss: 0.6778 - val_mae: 0.3645
Epoch 3/50
113/113 ━━━━━━━━━━━━━━━━━━━━ 41s 277ms/step - loss: 0.5929 - mae: 0.3153 - val_loss: 0.4975 - val_mae: 0.3652
Epoch 4/50
113/113 ━━━━━━━━━━━━━━━━━━━━ 42s 239ms/step - loss: 0.4343 - mae: 0.3149 - val_loss: 0.3849 - val_mae: 0.3683
Epoch 5/50
113/113 ━━━━━━━━━━━━━━━━━━━━ 49s 255ms/step - loss: 0.3285 - mae: 0.3123 - val_loss: 0.3034 - val_mae: 0.3586
Epoch 6/50
113/113 ━━━━━━━━━━━━━━━━━━━━ 54s 324ms/step - loss: 0.2565 - mae: 0.3010 - val_loss: 0.2411 - val_mae: 0.3336
Epoch 7/50
113/113 ━━━━━━━━━━━━━━━━━━━━ 43s 291ms/step - loss: 0.2065 - mae: 0.2894 - val_loss: 0.2055 - val_mae: 0.2856
Epoch 8/50
113/113 ━━━━━━━━━━━━━━━━━━━━ 54s 354ms/step - loss: 0.1644 - mae: 0.2538 - val_loss: 0.1604 - val_mae: 0.2692
Epoch 9/50
113/113 ━━━━━━━━━━━━

In [70]:
Photo_predictions = Photo_model.predict(X_test_windowed)
Photo_predictions

11/11 ━━━━━━━━━━━━━━━━━━━━ 44s 2s/step


array([[0.9525144 ],
       [0.82304287],
       [0.7715937 ],
       [0.35340005],
       [0.47553745],
       [0.98211807],
       [1.0311407 ],
       [0.78704476],
       [0.6232111 ],
       [0.66775155],
       [0.67811286],
       [0.89676887],
       [0.6977565 ],
       [0.70742834],
       [0.4728002 ],
       [0.64378774],
       [0.6839179 ],
       [0.47063363],
       [0.04632912],
       [0.33674526],
       [0.39373282],
       [0.7967821 ],
       [0.84206843],
       [0.6131705 ],
       [0.4070351 ],
       [0.42673713],
       [0.7569424 ],
       [0.49134055],
       [0.80527455],
       [0.67609   ],
       [0.43713766],
       [0.53601116],
       [0.6870822 ],
       [0.39492196],
       [0.04307859],
       [0.04692937],
       [0.05140426],
       [0.7032567 ],
       [0.7706185 ],
       [0.7869022 ],
       [0.7615446 ],
       [0.5976004 ],
       [0.49246076],
       [0.34301025],
       [0.06125217],
       [0.15350443],
       [0.38795078],
       [0.683

In [71]:
Photo_predictions = y_scaler.inverse_transform(Photo_predictions.reshape(-1, 1))
Photo_predictions = np.clip(Photo_predictions, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

In [72]:
Photo_resultados = pd.DataFrame(Photo_predictions, index = y_test_windowed.index, columns=["Photo"])

In [73]:
predicciones["Photo"] = Photo_resultados["Photo"]
predicciones

,Generación,LightGBM,Random Forest,CTNET,Forecast,Photo
13956,29864.0,29802.736495,29306.46,NaN,NaN,NaN
13960,29081.0,29095.19111,29272.7725,NaN,NaN,NaN
13961,28659.0,29136.770699,29224.675,NaN,NaN,NaN
13962,26410.0,28333.916793,28923.955,NaN,NaN,NaN
13963,17165.0,18488.082907,20188.37,NaN,NaN,NaN
...,...,...,...,...,...,...
18250,18014.0,21351.153925,20554.05,18707.601562,5487.743164,7901.168945
18251,23010.0,22831.557606,24302.920607,23466.720703,19692.076172,20805.576172
18252,26156.0,24912.539517,25643.327875,22903.962891,7768.808105,24043.478516
18285,0.0,0.0,0.0,20785.935547,2999.348389,16784.800781


In [74]:
print("LightGBM")
print(f"MAE: {mean_absolute_error(predicciones['Generación'], predicciones['LightGBM']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones['Generación'], predicciones['LightGBM'])):.4f}")
print(f"R²: {r2_score(predicciones['Generación'], predicciones['LightGBM']):.4f}")
print("Random Forest")
print(f"MAE: {mean_absolute_error(predicciones['Generación'], predicciones['Random Forest']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones['Generación'], predicciones['Random Forest'])):.4f}")
print(f"R²: {r2_score(predicciones['Generación'], predicciones['Random Forest']):.4f}")
print("CTNET")
print(f"MAE: {mean_absolute_error(predicciones.loc[~predicciones['CTNET'].isna(),'Generación'], predicciones.loc[~predicciones['CTNET'].isna(),'CTNET']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones.loc[~predicciones['CTNET'].isna(),'Generación'], predicciones.loc[~predicciones['CTNET'].isna(),'CTNET'])):.4f}")
print(f"R²: {r2_score(predicciones.loc[~predicciones['CTNET'].isna(),'Generación'], predicciones.loc[~predicciones['CTNET'].isna(),'CTNET']):.4f}")
print("Forecast")
print(f"MAE: {mean_absolute_error(predicciones.loc[~predicciones['Forecast'].isna(),'Generación'], predicciones.loc[~predicciones['Forecast'].isna(),'Forecast']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones.loc[~predicciones['Forecast'].isna(),'Generación'], predicciones.loc[~predicciones['Forecast'].isna(),'Forecast'])):.4f}")
print(f"R²: {r2_score(predicciones.loc[~predicciones['Forecast'].isna(),'Generación'], predicciones.loc[~predicciones['Forecast'].isna(),'Forecast']):.4f}")
print("Photovoltaic")
print(f"MAE: {mean_absolute_error(predicciones.loc[~predicciones['Photo'].isna(),'Generación'], predicciones.loc[~predicciones['Photo'].isna(),'Photo']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones.loc[~predicciones['Photo'].isna(),'Generación'], predicciones.loc[~predicciones['Photo'].isna(),'Photo'])):.4f}")
print(f"R²: {r2_score(predicciones.loc[~predicciones['Photo'].isna(),'Generación'], predicciones.loc[~predicciones['Photo'].isna(),'Photo']):.4f}")

LightGBM
MAE: 1180.6107
RMSE: 1838.9254
R²: 0.9682
Random Forest
MAE: 1137.5189
RMSE: 1896.5673
R²: 0.9661
CTNET
MAE: 7067.9263
RMSE: 9133.8398
R²: 0.1849
Forecast
MAE: 7534.2633
RMSE: 9745.5976
R²: 0.0720
Photovoltaic
MAE: 7003.3658
RMSE: 8643.8998
R²: 0.2700


In [75]:
# Seleccionar las columnas desde "LightGBM" en adelante
columnas_nuevas = predicciones.loc[:, "LightGBM":]

# Unir con `datos` usando el índice, manteniendo todo en `datos`
datos = datos.merge(columnas_nuevas, left_index=True, right_index=True, how='left')

# Ver resultado
datos.head()


,Fecha,Generación,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Cluster KMeans,Cluster GMM,Generacion_prev_hour,Generacion_prev_day,LightGBM,Random Forest,CTNET,Forecast,Photo
24,2022-09-02 00:00:00,0.0,19,6,76,0,4,15,0,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN
25,2022-09-02 01:00:00,0.0,18,7,81,0,4,15,1,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN
26,2022-09-02 02:00:00,0.0,18,7,84,0,4,15,2,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN
27,2022-09-02 03:00:00,0.0,18,7,86,0,4,15,3,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN
28,2022-09-02 04:00:00,0.0,17,7,86,0,4,15,4,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN


## X_train para hacer análisis de sobreajuste

In [76]:
predicciones_train = y_train.copy()

In [77]:
LightGBM_predictions_train = LightGBM_model.predict(X_train_scaled_df)
LightGBM_predictions_train = y_scaler.inverse_transform(LightGBM_predictions_train.reshape(-1, 1))
LightGBM_predictions_train = np.clip(LightGBM_predictions_train, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000
LightGBM_resultados = pd.DataFrame(LightGBM_predictions_train, index = y_train_scaled_df.index, columns=["LightGBM_train"])
predicciones_train["LightGBM_train"] = LightGBM_resultados["LightGBM_train"]

[LightGBM] [Warning] min_data_in_leaf is set=85, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=85


In [78]:
RandomForest_predictions_train = RF_model.predict(X_train_scaled_df)
RandomForest_predictions_train = y_scaler.inverse_transform(RandomForest_predictions_train.reshape(-1, 1))
RandomForest_predictions_train = np.clip(RandomForest_predictions_train, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000
RandomForest_resultados = pd.DataFrame(RandomForest_predictions_train, index = y_train_scaled_df.index, columns=["RandomForest_train"])
predicciones_train["RandomForest_train"] = RandomForest_resultados["RandomForest_train"]

In [79]:
CTNET_predictions_train = CTNET.predict(X_train_windowed)
CTNET_predictions_train = y_scaler.inverse_transform(CTNET_predictions_train.reshape(-1, 1))
CTNET_predictions_train = np.clip(CTNET_predictions_train, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000
CTNET_resultados = pd.DataFrame(CTNET_predictions_train, index = y_train_windowed.index, columns=["CTNET_train"])
predicciones_train["CTNET_train"] = CTNET_resultados["CTNET_train"]

57/57 ━━━━━━━━━━━━━━━━━━━━ 15s 135ms/step


In [80]:
Forecast_predictions_train = Forecast_model.predict(X_train_windowed)
Forecast_predictions_train = y_scaler.inverse_transform(Forecast_predictions_train.reshape(-1, 1))
Forecast_predictions_train = np.clip(Forecast_predictions_train, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000
Forecast_resultados = pd.DataFrame(Forecast_predictions_train, index = y_train_windowed.index, columns=["Forecast_train"])
predicciones_train["Forecast_train"] = Forecast_resultados["Forecast_train"]

57/57 ━━━━━━━━━━━━━━━━━━━━ 16s 165ms/step


In [81]:
Photo_predictions_train = Photo_model.predict(X_train_windowed)
Photo_predictions_train = y_scaler.inverse_transform(Photo_predictions_train.reshape(-1, 1))
Photo_predictions_train = np.clip(Photo_predictions_train, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000
Photo_resultados = pd.DataFrame(Photo_predictions_train, index = y_train_windowed.index, columns=["Photo_train"])
predicciones_train["Photo_train"] = Photo_resultados["Photo_train"]

57/57 ━━━━━━━━━━━━━━━━━━━━ 11s 77ms/step


In [82]:
predicciones_train

,Generación,LightGBM_train,RandomForest_train,CTNET_train,Forecast_train,Photo_train
35,17036.043251,18916.650982,18343.081911,NaN,NaN,NaN
36,27523.885172,23625.237046,26484.068664,NaN,NaN,NaN
227,26530.520060,22120.382702,25113.129493,NaN,NaN,NaN
228,28976.805233,27136.696426,28239.947956,NaN,NaN,NaN
275,27101.921674,23336.208873,26119.410594,NaN,NaN,NaN
...,...,...,...,...,...,...
11733,0.000000,813.079838,0.000000,20731.121094,21097.957031,19384.234375
11757,0.000000,126.613609,0.000000,6383.157715,4381.900879,3073.778809
11758,0.000000,180.801138,0.000000,8024.703613,1582.384155,6675.945801
11767,0.000000,0.000000,0.000000,8355.585938,6401.897461,8967.548828


In [83]:
print("LightGBM")
print(f"MAE: {mean_absolute_error(predicciones_train['Generación'], predicciones_train['LightGBM_train']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones_train['Generación'], predicciones_train['LightGBM_train'])):.4f}")
print(f"R²: {r2_score(predicciones_train['Generación'], predicciones_train['LightGBM_train']):.4f}")
print("Random Forest")
print(f"MAE: {mean_absolute_error(predicciones_train['Generación'], predicciones_train['RandomForest_train']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones_train['Generación'], predicciones_train['RandomForest_train'])):.4f}")
print(f"R²: {r2_score(predicciones_train['Generación'], predicciones_train['RandomForest_train']):.4f}")
print("CTNET")
print(f"MAE: {mean_absolute_error(predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'CTNET_train']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'CTNET_train'])):.4f}")
print(f"R²: {r2_score(predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'CTNET_train']):.4f}")
print("Forecast")
print(f"MAE: {mean_absolute_error(predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Forecast_train']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Forecast_train'])):.4f}")
print(f"R²: {r2_score(predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Forecast_train']):.4f}")
print("Photovoltaic")
print(f"MAE: {mean_absolute_error(predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Photo_train']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Photo_train'])):.4f}")
print(f"R²: {r2_score(predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Photo_train']):.4f}")

LightGBM
MAE: 1149.2564
RMSE: 2157.2697
R²: 0.9648
Random Forest
MAE: 421.1439
RMSE: 875.6741
R²: 0.9942
CTNET
MAE: 4924.6953
RMSE: 6616.9018
R²: 0.6707
Forecast
MAE: 5466.7562
RMSE: 7656.0800
R²: 0.5591
Photovoltaic
MAE: 5183.7918
RMSE: 6817.1150
R²: 0.6505


In [84]:
# Seleccionar las columnas desde "LightGBM" en adelante
columnas_nuevas = predicciones_train.loc[:, "LightGBM_train":]

# Unir con `datos` usando el índice, manteniendo todo en `datos`
datos = datos.merge(columnas_nuevas, left_index=True, right_index=True, how='left')

# Ver resultado
datos.head()


,Fecha,Generación,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Cluster KMeans,...,LightGBM,Random Forest,CTNET,Forecast,Photo,LightGBM_train,RandomForest_train,CTNET_train,Forecast_train,Photo_train
24,2022-09-02 00:00:00,0.0,19,6,76,0,4,15,0,Noche,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25,2022-09-02 01:00:00,0.0,18,7,81,0,4,15,1,Noche,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
26,2022-09-02 02:00:00,0.0,18,7,84,0,4,15,2,Noche,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,2022-09-02 03:00:00,0.0,18,7,86,0,4,15,3,Noche,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
28,2022-09-02 04:00:00,0.0,17,7,86,0,4,15,4,Noche,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [85]:
datos.to_excel("04.5_Predicciones_Conjunto_nublado GMM.xlsx", index=True)

## Guardamos los modelos

In [86]:
import joblib

# Guardar modelo LightGBM
joblib.dump(LightGBM_model, "4_5_LightGBM_model.pkl")

# Guardar modelo Random Forest
joblib.dump(RF_model, "4_5_RandomForest_model.pkl")


['4_5_RandomForest_model.pkl']

In [87]:
CTNET.save("4_5_CTNET_model.keras")
Forecast_model.save("4_5_Forecast_model.keras")
Photo_model.save("4_5_Photo_model.keras")